In [ ]:
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_core.messages import ToolMessageChunk
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage

import os
load_dotenv()
if os.environ["GOOGLE_API_KEY"]=="" :
    raise ValueError("Please set the GOOGLE_API_KEY environment variable in the .env file.")
else:
    print("GOOGLE_API_KEY is set.")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

In [ ]:
from pydantic import BaseModel, Field  
from typing import Literal

class llm_schema(BaseModel):
    category: Literal["insta", "twitter", "linkedin"] = Field(..., description="The social media platform to create a post for.")
    topic: str = Field(..., description="The topic for the social media post.")

In [ ]:
llm_with_schema = llm.with_structured_output(llm_schema)
llm_with_schema.invoke("i want to create a post for twitter about AI")

## State schema

In [ ]:
from typing import TypedDict, List
class graph_schema(TypedDict):
    input: str
    topic: str
    post: str
    category:str

In [ ]:
# from typing import TypedDict, List

# class graph_schema(TypedDict):
#     topic: str
#     insta: str
#     twitter: str
#     linkedin: str


In [ ]:
def decider_node(state: graph_schema) -> graph_schema:
    user_input = state['input']
    category = llm_with_schema.invoke(user_input).category
    topic = llm_with_schema.invoke(user_input).topic
    state['category'] = category
    state['topic'] = topic
    return state

def create_post_insta(state: graph_schema)-> graph_schema:
    # state = state.model_dump()
    topic = state['topic']
    post = llm.invoke(f"Create an instagram post about {topic}. Keep the tone casual and engaging.")
    state['insta'] = post
    return  {'post': state['insta']}

def create_post_twitter(state: graph_schema)-> graph_schema:
    # state = state.model_dump()
    topic = state['topic']
    post = llm.invoke(f"Create a Twitter post about {topic}. Keep the tone quick.")
    state['twitter'] = post
    return {'post': state['twitter']}

def create_post_linkedin(state: graph_schema)-> graph_schema:
    # state = state.model_dump()
    topic = state['topic']
    post = llm.invoke(f"Create a LinkedIn post about {topic}. Keep the tone professional.")
    state['linkedin'] = post
    return {'post': state['linkedin']}

In [ ]:
def condition(state: graph_schema) -> str:
    category = state['category']
    if category == "insta":
        return "create_post_insta"
    elif category == "twitter":
        return "create_post_twitter"
    elif category == "linkedin":
        return "create_post_linkedin"
    else:
        raise ValueError("Invalid category")

In [ ]:
from langgraph.graph import StateGraph, START, END
graph = StateGraph(graph_schema)
#adding nodes to the graph
graph.add_node("decider_node", decider_node)
graph.add_node("create_post_insta", create_post_insta)
graph.add_node("create_post_twitter", create_post_twitter)
graph.add_node("create_post_linkedin", create_post_linkedin)

#adding edges to the graph
graph.add_edge(START, "decider_node")
graph.add_conditional_edges("decider_node", condition, {"create_post_insta": "create_post_insta", "create_post_twitter": "create_post_twitter", "create_post_linkedin": "create_post_linkedin"})
# graph.add_edge("decider_node","create_post_insta")
# graph.add_edge("decider_node","create_post_twitter")
# graph.add_edge("decider_node","create_post_linkedin")
graph.add_edge("create_post_insta", END)
graph.add_edge("create_post_twitter", END)
graph.add_edge("create_post_linkedin", END)

route_graph = graph.compile()

from IPython.display import Image, display
Image(route_graph.get_graph().draw_mermaid_png())

In [ ]:
route_graph.invoke({"input": "Artificial Intelligence post to be posted on professional network",
                    "topic": "",
                    "post": "",
                    "category": ""}) 